# YOLOv8n — Cardboard Box Detection → CoreML

Dataset: **Cardboardbox detection v1** (Roboflow) — 124 train / 36 val / 18 test — 1 clase: `cardboard`

**⚠️ Antes de empezar:** Menú → Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 1. GPU check + instalar ultralytics ───────────────────────────────────
!pip install ultralytics coremltools -q

import torch
assert torch.cuda.is_available(), '❌ Sin GPU — cambiá el runtime a T4'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'   PyTorch: {torch.__version__}')

In [ ]:
# ── 2. Subir el zip del dataset ───────────────────────────────────────────
# Subí: "Cardboardbox detection.v1i.yolov8.zip"
from google.colab import files
import zipfile, os, yaml

print('Seleccioná el zip del dataset...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'Subido: {zip_name}')

os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')

print('\n✅ Dataset extraído')
!ls /content/dataset

In [ ]:
# ── 3. Corregir paths en data.yaml ────────────────────────────────────────
# Roboflow exporta paths como "../train/images" (relativos a donde está el yaml)
# Los corregimos a paths absolutos dentro de /content/dataset/

fixed_yaml = '/content/dataset/data.yaml'

with open(fixed_yaml) as f:
    cfg = yaml.safe_load(f)

cfg['train'] = '/content/dataset/train/images'
cfg['val']   = '/content/dataset/valid/images'
cfg['test']  = '/content/dataset/test/images'

with open(fixed_yaml, 'w') as f:
    yaml.dump(cfg, f)

# Verificar imágenes disponibles
import glob
for split, path in [('train', cfg['train']), ('val', cfg['val']), ('test', cfg['test'])]:
    n = len(glob.glob(f'{path}/*'))
    print(f'{split:5}: {n} imágenes → {path}')

print(f'\nClases ({cfg["nc"]}): {cfg["names"]}')

In [ ]:
# ── 4. Entrenamiento ──────────────────────────────────────────────────────
# Dataset pequeño (124 imgs) → augmentation fuerte + más epochs
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pesos preentrenados COCO

results = model.train(
    data      = fixed_yaml,
    epochs    = 200,
    imgsz     = 640,
    batch     = 16,
    device    = 0,
    patience  = 30,        # early stopping
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    # Augmentation fuerte para dataset pequeño
    mosaic    = 1.0,
    mixup     = 0.1,
    degrees   = 15.0,      # rotación (cajas en ángulo)
    scale     = 0.6,       # zoom variado
    shear     = 5.0,
    perspective=0.0005,
    flipud    = 0.2,
    fliplr    = 0.5,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    project   = '/content/runs',
    name      = 'box_detector',
    exist_ok  = True,
)

print('\n✅ Entrenamiento finalizado')

In [ ]:
# ── 5. Métricas finales ───────────────────────────────────────────────────
best = YOLO('/content/runs/box_detector/weights/best.pt')
metrics = best.val(data=fixed_yaml)

map50    = metrics.box.map50
map5095  = metrics.box.map
prec     = metrics.box.mp
recall   = metrics.box.mr

print(f'\n📊 Resultados en validación:')
print(f'   mAP@50:    {map50:.3f}  (>0.80 = bueno)')
print(f'   mAP@50-95: {map5095:.3f}')
print(f'   Precision: {prec:.3f}')
print(f'   Recall:    {recall:.3f}')

if map50 < 0.60:
    print('\n⚠️  mAP bajo. Considerá agregar más imágenes al dataset.')
elif map50 >= 0.80:
    print('\n✅ Modelo listo para producción.')

In [ ]:
# ── 6. Inferencia de prueba (opcional) ────────────────────────────────────
test_images = glob.glob('/content/dataset/test/images/*.jpg')[:4]
results_test = best.predict(test_images, conf=0.25, save=True, project='/content', name='preview')

from IPython.display import Image, display
for img_path in glob.glob('/content/preview/*.jpg'):
    display(Image(img_path, width=400))

In [ ]:
# ── 7. Exportar a CoreML ──────────────────────────────────────────────────
# nms=False → salida raw [1, 5, 8400] que parsea el código Swift
# (5 = 4 bbox + 1 clase; sin máscara de segmentación)
export_path = best.export(
    format = 'coreml',
    imgsz  = 640,
    nms    = False,
)
print(f'\n✅ Exportado: {export_path}')

In [ ]:
# ── 8. Comprimir y descargar ──────────────────────────────────────────────
import shutil

zip_out = '/content/box_detector.mlpackage.zip'
shutil.make_archive('/content/box_detector.mlpackage', 'zip', export_path)

size_mb = os.path.getsize(zip_out) / 1024 / 1024
print(f'Tamaño: {size_mb:.1f} MB')
print('Descargando...')
files.download(zip_out)

print()
print('📋 Próximos pasos:')
print('1. Descomprimir box_detector.mlpackage.zip')
print('2. Renombrar la carpeta a: box_detector.mlpackage')
print('3. Reemplazar en el repo: modules/lidar-box-measure/ios/coco128-yolo11n-seg.mlpackage/')
print('   con la nueva carpeta box_detector.mlpackage/')
print('4. Actualizar LidarBoxMeasure.podspec: cambiar nombre del recurso')
print('5. Actualizar loadModel() en BoxDetectionCoordinator.swift:')
print('   - Agregar "box_detector" a la lista de nombres')
print('   - Cambiar numSegMasks = 32 → numSegMasks = 0')
print('   - Cambiar numClasses = C - 4 - numSegMasks → numClasses = C - 4')

## Cambios necesarios en Swift después de entrenar

El nuevo `yolov8n` (detección pura) tiene salida `[1, 5, 8400]` (4 bbox + 1 clase).
El modelo actual `yolov11n-seg` tiene salida `[1, 116, 8400]` (4 + 80 clases + 32 máscaras).

### En `BoxDetectionCoordinator.swift`:

```swift
// ANTES:
let numSegMasks = 32
let numClasses  = C - 4 - numSegMasks   // C=116: 4+80+32

// DESPUÉS:
let numSegMasks = 0
let numClasses  = C - 4                 // C=5: 4+1
```

### En `loadModel()`:
```swift
let names = ["box_detector", "coco128-yolo11n-seg", ...]
```

### En `LidarBoxMeasure.podspec`:
```ruby
s.resources = ['ios/box_detector.mlpackage']
```